# mSWE-GNN (GPU) vs SFINCS (4-thread CPU) inference timing

GPU counterpart of `measure_ahr_inference_timing.ipynb` (which measures GNN CPU vs SFINCS CPU,
both on 4 dedicated cores). This notebook runs the GNN on GPU instead — the regime the original
mSWE-GNN paper's speed-up claim is actually about (GPU-parallel DL inference vs a CPU numerical
solver), rather than CPU vs CPU.

Mesh-building time is excluded on both sides (same convention as the paper): it's a one-off cost
shared by both methods, not a per-simulation cost. Only inference vs. solver run time is measured.

SFINCS timings are `Total simulation time` [s] (solver only, excludes I/O/init, 4 threads), read
from the summary block at the end of each event's own
`database/raw_datasets_ahr/Simulations/.../sfincs.log`.


In [ ]:
import os, sys
REPO_ROOT = os.path.dirname(os.getcwd())
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

import time
import torch
import wandb
import lightning as L
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader

from utils.load import read_config
from utils.miscellaneous import get_model, fix_dict_in_config
from utils.dataset import create_model_dataset, get_temporal_test_dataset_parameters, to_temporal_dataset
from training.train import LightningTrainer

torch.backends.cudnn.deterministic = True
torch.set_float32_matmul_precision('high')

# CPU threading doesn't matter much here (the GNN runs on GPU) -- only affects
# host-side data loading/transfer.
N_THREADS = int(os.environ.get('SLURM_CPUS_PER_TASK', 4))
os.environ['OMP_NUM_THREADS'] = str(N_THREADS)
torch.set_num_threads(N_THREADS)
torch.set_num_interop_threads(1)

CONFIG     = 'config_best_sweep_multisim.yaml'
CHECKPOINT = os.path.join(REPO_ROOT, 'results', 'best_sweep_multisim.h5')

# tag -> (test dataset name, SFINCS "Total simulation time" [s] from sfincs.log)
SCENARIOS = {
    'q050': ('ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_q050', 7.503),
    'q075': ('ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_q075', 8.307),
    'q125': ('ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_q125', 9.502),
    'q150': ('ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_q150', 10.636),
    'q200': ('ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_q200', 10.824),
    'q300': ('ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_q300', 13.170),
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
assert device.type == 'cuda', 'No GPU visible -- submit via run_measure_ahr_timing_gpu_hal8.sh (partition=gpu)'

## Load model once

Architecture (mesh template, `num_scales`, `hid_features`, `K`) is the same across every event, so the model only needs to be built and loaded once.

In [ ]:
cfg = read_config(CONFIG)
wandb.init(mode='disabled', project='mswe-gnn', config=cfg)
fix_dict_in_config(wandb)
config = wandb.config

temporal_test_dataset_parameters = get_temporal_test_dataset_parameters(
    config, config.temporal_dataset_parameters
)

_, _, test_dataset, scalers = create_model_dataset(
    scalers=config.scalers, device=device,
    **config.dataset_parameters,
    **config.selected_node_features,
    **config.selected_edge_features
)
temporal_dataset_probe = to_temporal_dataset(test_dataset, rollout_steps=-1, **temporal_test_dataset_parameters)
num_node_features = temporal_dataset_probe[0].x.size(-1)
num_edge_features = temporal_dataset_probe[0].edge_attr.size(-1)

model_parameters = dict(config.models)
model_type = model_parameters.pop('model_type')
if model_type == 'MSGNN':
    model_parameters['num_scales'] = test_dataset[0].mesh.num_meshes

_ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
_sd = _ckpt['state_dict']
_hid = _sd['model.edge_encoder.0.weight'].shape[0]

# K is per-scale (e.g. [1, 1, 1, 5, 4, 3, 2]), not a single value shared by every
# gnn_processor stage — must be detected per processor index, not just index 0.
_proc_ids = sorted(set(
    int(k.split('.')[2]) for k in _sd
    if k.startswith('model.gnn_processor.') and 'filter_matrix.' in k
))
_K_list = [
    sum(1 for k in _sd if f'model.gnn_processor.{i}.filter_matrix.' in k and k.endswith('.weight')) - 1
    for i in _proc_ids
]
if _hid != model_parameters.get('hid_features') or _K_list != model_parameters.get('K'):
    print(f'Checkpoint arch override: hid_features={_hid}, K={_K_list}')
    model_parameters['hid_features'] = _hid
    model_parameters['K'] = _K_list

model = get_model(model_type)(
    num_node_features=num_node_features,
    num_edge_features=num_edge_features,
    previous_t=temporal_test_dataset_parameters['previous_t'],
    device=device,
    **model_parameters
).to(device)

plmodule = LightningTrainer.load_from_checkpoint(
    CHECKPOINT, map_location=device,
    model=model,
    lr_info=config['lr_info'],
    trainer_options=config.trainer_options,
    temporal_test_dataset_parameters=temporal_test_dataset_parameters
)
model = plmodule.model.to(device)
model.eval()
print(f'Model loaded — epoch {_ckpt["epoch"]}')

trainer = L.Trainer(accelerator='gpu', devices=1, logger=False,
                     enable_progress_bar=False, enable_checkpointing=False)

In [ ]:
# untimed warm-up: the first trainer.predict() call eats one-time CPU/Lightning
# setup overhead that would otherwise unfairly inflate whichever scenario runs first
warmup_dataloader = DataLoader(temporal_dataset_probe, batch_size=len(temporal_dataset_probe), shuffle=False)
trainer.predict(plmodule, dataloaders=warmup_dataloader)
print('Warm-up predict done.')

## Time inference per event

In [ ]:
results = []
for tag, (dataset_name, sfincs_time) in SCENARIOS.items():
    cfg_i = read_config(CONFIG)
    cfg_i['dataset_parameters']['test_dataset_name'] = dataset_name
    wandb.init(mode='disabled', project='mswe-gnn', config=cfg_i)
    fix_dict_in_config(wandb)
    config_i = wandb.config

    _, _, test_dataset_i, _ = create_model_dataset(
        scalers=config_i.scalers, device=device,
        **config_i.dataset_parameters,
        **config_i.selected_node_features,
        **config_i.selected_edge_features
    )
    temporal_test_dataset = to_temporal_dataset(
        test_dataset_i, rollout_steps=-1, **temporal_test_dataset_parameters
    )
    test_dataloader = DataLoader(temporal_test_dataset, batch_size=len(temporal_test_dataset), shuffle=False)

    start_time = time.time()
    trainer.predict(plmodule, dataloaders=test_dataloader)
    gnn_time = (time.time() - start_time) / len(temporal_test_dataset)

    speed_up = sfincs_time / gnn_time
    results.append((tag, sfincs_time, gnn_time, speed_up))
    print(f'{tag}: SFINCS={sfincs_time:.3f}s  GNN={gnn_time:.4f}s  speed-up={speed_up:.1f}x')

## Summary table and plot

In [ ]:
print(f'{"scenario":10s}{"SFINCS [s]":>12s}{"GNN [s]":>12s}{"speed-up":>12s}')
for tag, sfincs_time, gnn_time, speed_up in results:
    print(f'{tag:10s}{sfincs_time:12.3f}{gnn_time:12.4f}{speed_up:12.1f}')

avg_speedup = sum(r[3] for r in results) / len(results)
print(f'\nAverage speed-up: {avg_speedup:.1f}x')

In [ ]:
tags = [r[0] for r in results]
speedups = [r[3] for r in results]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(tags, speedups)
ax.axhline(1, color='k', linestyle=':', linewidth=1)
ax.set_ylabel('Speed-up (SFINCS time / GNN time)')
ax.set_title('mSWE-GNN vs SFINCS inference speed-up')
plt.tight_layout()